## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (StratifiedKFold, cross_validate)
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

## Loading Data

In [ ]:
data_path = "datasets"
contract_path = os.path.join(data_path, "contract.csv")
internet_path = os.path.join(data_path, "internet.csv")
personal_path = os.path.join(data_path, "personal.csv")
phone_path = os.path.join(data_path, "phone.csv")

df_contract = pd.read_csv(contract_path)
df_internet = pd.read_csv(internet_path)
df_personal = pd.read_csv(personal_path)
df_phone = pd.read_csv(phone_path)

## Stage 1:
1. *¿Cómo planeas unir los datos y qué harás con los valores nulos generados?*
- Go to "Merge Dataframes and target creation" section.
2. *¿Cuál será tu variable objetivo y qué tipo de problema de Machine Learning resolverás?*
- Go to "Create target" section.
3. *¿Qué pasos de preprocesamiento (codificación categórica, fechas, etc.) consideras necesarios y sobre que variables?*
- Go to Stage 2.2 Preprocessing for full details.

4. *¿Qué modelos planeas entrenar?*
- Go to Models Stage 2.4 to review all models which were trained.

## Stage 2.1: Exploratory Data Analysis and Data Wrangling

### contract dataframe

In [ ]:
df_contract.info()

In [ ]:
df_contract.sample(10)

Things to check in general in this dataframe.
- Check for duplicate rows in the dataframe

Things to check within this dataframe;

- custormerID: if there are nulls or repeated IDs
- BeginDate: if there are nulls or incorrect dates (greater than today or earlier than a reasonable past date). Change format to pd.Datetime
- EndDate: if there are nulls or incorrect dates (earlier than BeginDate or later than today); transform "No" into pd.NaT value
- Type: check uniques in order to transform into a categorical variable
- PaperlessBilling: check uniques in order to transform into a categorical variable
- PaymentMethod: check uniques in order to transform into a categorical variable
- MonthlyCharges: check for nulls and incorrect values (negative or excessively high)
- TotalCharges: check for nulls and incorrect values (negative or excessively high)


#### General

In [ ]:
# Check missing /null values and duplicates
missing = df_contract.isnull().sum()
duplicates = df_contract.duplicated().sum()
print("Missing values:\n", missing)
print("Duplicate rows:", duplicates)

#### customerID

In [ ]:
# Check for duplicate customerID in the contract dataframe
sum(df_contract["customerID"].value_counts() > 1)

#### BeginDate

In [ ]:
# Convert BeginDate to datetime format
df_contract["BeginDate"] = pd.to_datetime(df_contract["BeginDate"], errors="coerce", format="%Y-%m-%d")
df_contract.sample(10)

#### EndDate

In [ ]:
# Convert EndDate to datetime format
df_contract["EndDate"] = pd.to_datetime(df_contract["EndDate"], errors="coerce", format="%Y-%m-%d %H:%M:%S")
df_contract.sample(10)

#### Type

In [ ]:
# Check unique types
df_contract["Type"].unique()

#### PaperlessBilling

In [ ]:
# Check unique types
df_contract["PaperlessBilling"].unique()

#### PaymentMethod

In [ ]:
# Check unique types
df_contract["PaymentMethod"].unique()

#### MonthlyCharges

In [ ]:
# check for negative or extreamly high values
min = df_contract[df_contract["MonthlyCharges"] < 0]
high = df_contract[df_contract["MonthlyCharges"] > 1000]
print("Negative MonthlyCharges:\n", min)
print("Extremely High MonthlyCharges:\n", high)

#### TotalCharges

In [ ]:
# This column is str; it will be changed to numerical
df_contract["TotalCharges"] = pd.to_numeric(df_contract["TotalCharges"], errors="coerce")

# check if there are nulls
df_contract[df_contract["TotalCharges"].isna()]
idx = df_contract[df_contract["TotalCharges"].isna()].index

As seen, the "TotalCharges" column had some non-numeric values which were converted to NaN.

This probably is due the end of the timelapse of the dataframe, where some entries have not accumulated any charges yet.

Therefore, it might be reasonable to fill these NaN values with the same monthly value, assuming no charges have been accumulated yet; but prio to do so, I'll explore which is the most recent date in the dataframe

In [ ]:
df_contract["BeginDate"].min()

In [ ]:
df_contract["EndDate"].min()

In [ ]:
# Since my guess was correct: Replacing NaN values in the "TotalCharges" column with the value MonthlyCharges
df_contract["TotalCharges"] = df_contract["TotalCharges"].fillna(df_contract["MonthlyCharges"])

# Explore same rows after filling NaN values
df_contract.iloc[idx]

In [ ]:
# check for negative or extreamly high values
min = df_contract[df_contract["TotalCharges"] < 0]
high = df_contract[df_contract["TotalCharges"] > 5000]
print("Negative TotalCharges:\n", min)
print("Extremely High TotalCharges:\n", high)

### internet dataframe

In [ ]:
df_internet.info()

In [ ]:
df_internet.sample(10)

Things to check in general in this dataframe.
- Check for duplicate rows in the dataframe

Things to check within this dataframe;

- custormerID: if there are nulls or repeated IDs
- InternetService: check uniques in order to transform into a categorical variable
- OnlineSecurity: check uniques in order to transform into a categorical variable
- OnlineBackup: check uniques in order to transform into a categorical variable
- TechSupport: check uniques in order to transform into a categorical variable
- StreamingTV: check uniques in order to transform into a categorical variable
- StreamingMovies: check uniques in order to transform into a categorical variable


#### General

In [ ]:
# Check for missing values
missings = df_internet.isna().sum()
# Check for duplicate rows
duplited = df_internet.duplicated().sum()
print("Missing values per column:\n", missings)
print("Duplicate rows in the dataframe: ", duplited)

#### InternetService

In [ ]:
df_internet['InternetService'].unique()

#### OnlineSecurity


In [ ]:
df_internet['OnlineSecurity'].unique()

#### OnlineBackup 


In [ ]:
df_internet['OnlineBackup'].unique()    

#### DeviceProtection

In [ ]:
df_internet['DeviceProtection'].unique()

#### TechSupport

In [ ]:
df_internet['TechSupport'].unique()

#### StreamingTV


In [ ]:
df_internet['StreamingTV'].unique()

#### StreamingMovies

In [ ]:
df_internet['StreamingMovies'].unique()

### personal dataframe

In [ ]:
df_personal.info()

In [ ]:
df_personal.sample(10)

#### General

In [ ]:
# Missing Values
missings = df_personal.isna().sum()
# Duplicates
duplicates = df_personal.duplicated().sum()
print("Missing Values:\n", missings)
print("Duplicates:", duplicates)

#### gender

In [ ]:
df_personal["gender"].unique()

#### SeniorCitizen

In [ ]:
df_personal["SeniorCitizen"].unique()

In [ ]:
# As seen; SeniorCitizen should be Yes No column as the other categorical columns
df_personal["SeniorCitizen"] = df_personal["SeniorCitizen"].replace({1: 'Yes', 0: 'No'})

In [ ]:
df_personal["SeniorCitizen"].unique()

#### Partner

In [ ]:
df_personal["Partner"].unique()

#### Dependents

In [ ]:
df_personal["Dependents"].unique()

### phone dataframe

In [ ]:
df_phone.info()

In [ ]:
df_phone.sample(10)

#### General

In [ ]:
# Missing Values
missings = df_phone.isna().sum()
# Duplicates
duplicates = df_phone.duplicated().sum()
print("Missing Values:\n", missings)
print("Duplicates:", duplicates)

#### MultipleLines

In [ ]:
df_phone["MultipleLines"].unique()

### Conclusions of EDA

- In general all dataframes are correct and do not include missing or null values.
- There are a few columns names in the dataframes that do not follow the PascalCase; this issue will be fixed when mergin dataframes
- The number of rows is different in each datataframe; contract has ; internet has ; personal has ; and phone has . This means that some customers have not contracted some services.
- The column SeniorCitizen in the dataframe personal

## Merge Dataframes and target creation

This shall be made using customerID as common (anchor) for all dataframes.

Strategy to fix NaN o missing values when merging:
- Missing "Yes/No" values will be changed to "No"
- Dataframes don't have same length; it is infered that customer have not contracted those services

In [ ]:
df_merged = df_contract.merge(df_internet, on="customerID", how="outer").merge(df_personal, on="customerID", how="outer").merge(df_phone, on="customerID", how="outer")
df_merged.reset_index(drop=True, inplace=True)

In [ ]:
# Check merge and look for missing values and duplicates
df_merged.info()

In [ ]:
duplicates = df_merged.duplicated().sum()
print("Duplicate rows:", duplicates)

In [ ]:
columns_to_fix = df_merged.columns
columns_to_fix = columns_to_fix.drop(["customerID","BeginDate","EndDate","Type"])
columns_to_fix

In [ ]:
for column in columns_to_fix:
    df_merged[column]= df_merged[column].replace(to_replace=np.nan, value="No")

In [ ]:
df_merged.info()

Conclusions:

- The merged dataset has been cleaned by replacing missing values with "No" for relevant columns.
- No duplicated rows were found.
- Further analysis can now be conducted on this cleaned and prepared dataset.

## Models Training

### Stage 2.2: Preprocessing

#### Create target

In [ ]:
# Prior to training models; the target variable needs to be clearly defined.
# Is the column EndDate has a valid date value; the client has churned. If it is missing or NaN, the client is still active.
df_merged['Churned'] = df_merged['EndDate'].notna()
df_merged[['EndDate', 'Churned']].head()

#### Explore balance of the target variable


In [ ]:
df_merged['Churned'].value_counts()/len(df_merged)*100

It can be seen that there is a moderate class imbalance in the target variable; since it is not severe class_weight should be enough to handle it during model training.

#### Keep relevant columns and add new columns as needed


In [ ]:
# customerID: irrelevant for model training
# BeginDate: Should be transformed into the number of days passed since it is a customer
# EndDate: irrelevant (Churned column already captures this information)
# MonthlyCharges: Should be normalized before feeding into the model
# TotalCharges: Should be normalized before feeding into the model

df_model = df_merged.copy()
df_model['BeginDate'] = pd.to_datetime(df_model['BeginDate'])

def client_since_days(row: pd.Series) -> int:
    """Calculate the number of days a client has been a customer. Using the beginning and end dates.
    if the 'EndDate' is not available, it assumes the reference day as "2020-02-01" to calculate the number of days.

    Args:
        row (pd.Series): A row of the DataFrame containing 'BeginDate' and 'EndDate'.

    Returns:
        int: The number of days the client has been a customer.
    """
    if pd.notna(row["EndDate"]):
        return (row["EndDate"] - row['BeginDate']).days
    else:
        return (pd.to_datetime("2020-02-01") - row['BeginDate']).days

df_model['DaysAsCustomer'] = df_model.apply(client_since_days, axis=1)

In [ ]:
# Drop columns that are no longer needed for the model training
df_model = df_model.drop(columns=["customerID",'EndDate'])
df_model = df_model.drop(columns=['BeginDate'])

In [ ]:
df_model.info()

#### Set features and target independent variables


In [ ]:
features = df_model.drop(columns=["Churned"])
target = df_model["Churned"]

#### Set columns as categorical and numerical

Columns types object, will be transformed into categorical using One Hot Encoder tecnique.

Columns types numeric, will be standarized using standard scaler.

In [ ]:
cat_cols = ["Type",
            "PaperlessBilling",
            "PaymentMethod",
            "InternetService",
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies",
            "gender",
            "SeniorCitizen",
            "Partner",
            "Dependents",
            "MultipleLines"]

num_cols = ["MonthlyCharges",
            "TotalCharges",
            "DaysAsCustomer"]

# Create the preprocessor
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') 
scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, cat_cols),
        ("num", scaler, num_cols)
    ]
).set_output(transform="pandas")



#### K-Folds

In [ ]:
# Define the Stratified K-Fold cross-validator for model evaluation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=12345
)

### Models

#### Stage 2.3: Baseline models (no balanced class weights)

##### Logistic Regression

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=12345
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

scores = cross_validate(
    model_pipeline,
    features,
    target,
    cv=skf,
    scoring=["roc_auc","recall"]
)

print("ROC-AUC per fold:", scores['test_roc_auc'])
print("Recall per fold:", scores['test_recall'])

print("Average ROC-AUC:", scores['test_roc_auc'].mean())
print("Average Recall:", scores['test_recall'].mean())

##### Decision Tree

In [ ]:
best_depth = 0
best_leaf_size = 0
depths = [5, 7, 10, 15]
leaf_sizes = [5, 10, 20, 50]

best_roc_auc = float("-inf")

for depth in depths:
    for leaf_size in leaf_sizes:
        model = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=leaf_size,
            random_state=12345
        )

        model_pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model)
            ]   
        )

        scores = cross_validate(
        model_pipeline,
        features,
        target,
        cv=skf,
        scoring=["roc_auc","recall"]
        )

        mean_roc_auc = scores["test_roc_auc"].mean()
        best_recall = scores["test_recall"].mean()
        if mean_roc_auc > best_roc_auc:
            best_roc_auc = mean_roc_auc
            best_score = scores
            best_depth = depth
            best_leaf_size = leaf_size


print("Best depth:", best_depth)
print("Best leaf size:", best_leaf_size)
print("ROC-AUC per fold:", best_score['test_roc_auc'])
print("Recall per fold:", best_score['test_recall'])
print("Average ROC-AUC:", best_score['test_roc_auc'].mean())
print("Average Recall:", best_score['test_recall'].mean())

#### Stage 2.4: Optimization (models balanced)

In [ ]:
results = {}

##### Logistic Regression

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=12345
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

scores = cross_validate(
    model_pipeline,
    features,
    target,
    cv=skf,
    scoring=["roc_auc","recall"]
)

print("ROC-AUC per fold:", scores['test_roc_auc'])
print("Recall per fold:", scores['test_recall'])

print("Average ROC-AUC:", scores['test_roc_auc'].mean())
print("Average Recall:", scores['test_recall'].mean())

results["logistic"] = {
        "roc-auc": scores['test_roc_auc'].mean(),
        "recall": scores['test_recall'].mean()
}

##### Decision Tree

In [ ]:
best_depth = 0
best_leaf_size = 0
depths = [5, 7, 10, 15]
leaf_sizes = [5, 10, 20, 50]

best_roc_auc = float("-inf")

for depth in depths:
    for leaf_size in leaf_sizes:
        model = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=leaf_size,
            class_weight='balanced',
            random_state=12345
        )

        model_pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model)
            ]   
        )

        scores = cross_validate(
        model_pipeline,
        features,
        target,
        cv=skf,
        scoring=["roc_auc","recall"]
        )

        mean_roc_auc = scores["test_roc_auc"].mean()
        best_recall = scores["test_recall"].mean()
        if mean_roc_auc > best_roc_auc:
            best_roc_auc = mean_roc_auc
            best_score = scores
            best_depth = depth
            best_leaf_size = leaf_size


print("Best depth:", best_depth)
print("Best leaf size:", best_leaf_size)
print("ROC-AUC per fold:", best_score['test_roc_auc'])
print("Recall per fold:", best_score['test_recall'])
print("Average ROC-AUC:", best_score['test_roc_auc'].mean())
print("Average Recall:", best_score['test_recall'].mean())

results["decision_tree"] = {
        "best_depth": best_depth,
        "best_leaf_size": best_leaf_size,
        "roc-auc": best_score['test_roc_auc'].mean(),
        "recall": best_score['test_recall'].mean()
}

##### Random Forest

In [ ]:
depths = [5, 7, 10, 15]
leaf_sizes = [2, 3, 4, 5, 10]
n_estimators = [100, 200, 300]

best_roc_auc = float('-inf')  # Initialize best ROC-AUC score to negative infinity
best_recall = float('-inf')  # Initialize best Recall score to negative infinity
best_n_estimator = 0

for depth in depths:
    for leaf_size in leaf_sizes:
        for n_estimator in n_estimators:
            model = RandomForestClassifier(
                n_estimators=n_estimator, 
                max_depth=depth, 
                min_samples_leaf=leaf_size, 
                random_state=12345
            )

            model_pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model)
                ]   
            )

            scores = cross_validate(
                model_pipeline,
                features,
                target,
                cv=skf,
                scoring=["roc_auc","recall"]
            )

            mean_roc_auc = scores["test_roc_auc"].mean()
            mean_recall = scores["test_recall"].mean()

            if mean_roc_auc > best_roc_auc:
                best_roc_auc = mean_roc_auc
                best_recall = mean_recall
                best_score = scores
                best_depth = depth
                best_leaf_size = leaf_size
                best_n_estimator = n_estimator


print("Best depth:", best_depth)
print("Best leaf size:", best_leaf_size)
print("Best number of estimators:", best_n_estimator)
print("ROC-AUC per fold:", best_score['test_roc_auc'])
print("Recall per fold:", best_score['test_recall'])
print("Average ROC-AUC:", best_score['test_roc_auc'].mean())
print("Average Recall:", best_score['test_recall'].mean())

results["random_forest"] = {
        "best_depth": best_depth,
        "best_leaf_size": best_leaf_size,
        "best_n_estimator": best_n_estimator,
        "roc-auc": best_score['test_roc_auc'].mean(),
        "recall": best_score['test_recall'].mean()
}

##### CatBoost

In [ ]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    random_seed=12345,
    verbose=False)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

scores = cross_validate(
        model_pipeline,
        features,
        target,
        cv=skf,
        scoring=["roc_auc","recall"]
)
print("ROC-AUC per fold:", scores['test_roc_auc'])
print("Recall per fold:", scores['test_recall'])

print("Average ROC-AUC:", scores['test_roc_auc'].mean())
print("Average Recall:", scores['test_recall'].mean())

results["catboost"] = {
    "roc-auc": scores['test_roc_auc'].mean(),
    "recall": scores['test_recall'].mean()
}

##### LightGBM

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    scale_pos_weight=2.77, # relation between classes Not churned / Churned
    verbosity=-1,
    random_state=12345
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

scores = cross_validate(
        model_pipeline,
        features,
        target,
        cv=skf,
        scoring=["roc_auc","recall"]
)

print("ROC-AUC per fold:", scores['test_roc_auc'])
print("Recall per fold:", scores['test_recall'])

print("Average ROC-AUC:", scores['test_roc_auc'].mean())
print("Average Recall:", scores['test_recall'].mean())

results["lightgbm"] = {
    "roc-auc": scores['test_roc_auc'].mean(),
    "recall": scores['test_recall'].mean()
}


## Best Model Evaluation

In [ ]:
roc_auc_scores = []
recall_scores = []
model_name = []
for key, model in results.items():
        roc_auc_scores.append(round(model["roc-auc"], 2))
        recall_scores.append(round(model["recall"], 2))
        model_name.append(key)

df_results = pd.DataFrame({
    "model": model_name,
    "roc-auc": roc_auc_scores,
    "recall": recall_scores
})

df_results

## Stage 3: Informe de Solución
*Escribe aquí tu informe final para el equipo de negocio. Asegúrate de responder:*
1. *¿Qué modelo elegiste finalmente y por qué?*
- I chose LightGBM because it offers the best roc-auc and recall metrics in contrast with the other models.
2. *¿Cuáles fueron las métricas finales (AUC-ROC y Recall) en el conjunto de prueba?*
- I did not perform a train_test_split; I used a K-fold (cross-validation) approach to solve this problem in order to do a more "realistic" evaluation of this problem. The mean AUC-ROC metric for the best model was 0.90 and the mean recall was 0.78
3. *En términos de negocio: ¿Qué significa tu valor de Recall? ¿Cómo impactaría tu modelo en la retención de clientes si el equipo de marketing lo utiliza hoy?*
- The recall shows how good is the model detecting the customers that are in risk to churn. For example; a recall of 0.80 could be read as "For every 100 customers, the model identified 80 that churned; and the missing 20 to reach the 100 customers were unrecognized by the model" 
For these reason it was really important to verify that classes were balanced or to use methods to balance them. When the decision tree model was run using unbalanced data, the recall was around 50% but the roc-auc metric was around 0.84; so using these model to predict churn clients will miss lead the marketing team into wrong retention strategies or promotions to the customers.



### Conclusions


- The traning of models with unbalanced clases was perform showing that a high roc-auc metric could be obtained but with a recall ~0.55 and 0.49 for logistic and decition tree models. This means that although the model is correct; around 50% of the true positives are not being caught by these models.
- After class balance; it could be observe that the best model is LightGBM with a roc-auc of 0.9 and a recall of 0.78; which is a great metric. The use of SMOTE tecnique or upsampling for the lowest class was no needed. A test using SMOTE showed no significant improvement in the models (this part is not shown in this notebook as it does not delivers value in the discusion of results) 
- The logistic and decision tree models worked surpricely well; offering great results with low computational cost.